In [1]:
!pip install pandas numpy matplotlib scikit-learn imbalanced-learn

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load Kaggle dataset
df = pd.read_csv("Dataset-SA.csv")

# Display dataset information
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

# Keep only positive and negative reviews
df = df[df["Sentiment"].str.lower().isin(["positive", "negative"])].copy()

# Convert sentiment into numerical values
df["Sentiment"] = df["Sentiment"].str.lower().map({
    "negative": 0,
    "positive": 1
})

# Remove missing reviews
df = df.dropna(subset=["Review", "Sentiment"])

# Keep a sample for faster execution
df = df.sample(
    n=min(10000, len(df)),
    random_state=42
)

# Display final dataset
print("\nPrepared Dataset:")
print(df.head())

print("\nSentiment Distribution:")
print(df["Sentiment"].value_counts())

Dataset Shape: (205052, 6)

Columns:
['product_name', 'product_price', 'Rate', 'Review', 'Summary', 'Sentiment']

First 5 Rows:
                                        product_name product_price Rate  \
0  Candes 12 L Room/Personal Air Cooler??????(Whi...          3999    5   
1  Candes 12 L Room/Personal Air Cooler??????(Whi...          3999    5   
2  Candes 12 L Room/Personal Air Cooler??????(Whi...          3999    3   
3  Candes 12 L Room/Personal Air Cooler??????(Whi...          3999    1   
4  Candes 12 L Room/Personal Air Cooler??????(Whi...          3999    3   

            Review                                            Summary  \
0           super!  great cooler excellent air flow and for this p...   
1          awesome              best budget 2 fit cooler nice cooling   
2             fair  the quality is good but the power of air is de...   
3  useless product                  very bad product its a only a fan   
4             fair                                      

# Task 1 – GridSearchCV with Random Forest (Product Reviews)

In [4]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Separate features and target
X = df["Review"].astype(str)
y = df["Sentiment"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=2000)),
    ("rf", RandomForestClassifier(random_state=42))
])

# Hyperparameter grid
param_grid = {
    "rf__n_estimators": [50, 100],
    "rf__max_depth": [10, 20, None]
}

# GridSearchCV
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

# Train model
grid_search.fit(X_train, y_train)

# Best model prediction
y_pred = grid_search.predict(X_test)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV F1 Score:")
print(grid_search.best_score_)

print("\nTest Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Best Parameters:
{'rf__max_depth': None, 'rf__n_estimators': 50}

Best CV F1 Score:
0.9780713836189318

Test Accuracy:
0.9625

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.81      0.86       274
           1       0.97      0.99      0.98      1726

    accuracy                           0.96      2000
   macro avg       0.94      0.90      0.92      2000
weighted avg       0.96      0.96      0.96      2000



# Task 2 – RandomizedSearchCV with SVM

In [5]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform

# Create SVM pipeline
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=2000)),
    ("scaler", StandardScaler(with_mean=False)),
    ("svm", SVC(kernel="rbf"))
])

# Parameter distribution
param_dist = {
    "svm__C": loguniform(0.1, 100),
    "svm__gamma": loguniform(0.001, 1)
}

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    svm_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

# Train model
random_search.fit(X_train, y_train)

# Predict test data
svm_pred = random_search.predict(X_test)

print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV F1 Score:")
print(random_search.best_score_)

print("\nTest Accuracy:")
print(accuracy_score(y_test, svm_pred))

print("\nClassification Report:")
print(classification_report(y_test, svm_pred))

Best Parameters:
{'svm__C': np.float64(1.3292918943162166), 'svm__gamma': np.float64(0.711447600934342)}

Best CV F1 Score:
0.9777941577973337

Test Accuracy:
0.9625

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.81      0.85       274
           1       0.97      0.99      0.98      1726

    accuracy                           0.96      2000
   macro avg       0.94      0.90      0.92      2000
weighted avg       0.96      0.96      0.96      2000



# Task 3 – SMOTE with GridSearchCV

In [6]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

# Check class distribution
print("Original Training Class Distribution:")
print(y_train.value_counts())

# Create pipeline with TF-IDF, SMOTE and Random Forest
smote_pipeline = ImbPipeline([
    ("tfidf", TfidfVectorizer(max_features=1000)),
    ("smote", SMOTE(random_state=42)),
    ("rf", RandomForestClassifier(random_state=42))
])

# Parameter grid
smote_grid = {
    "rf__n_estimators": [50, 100],
    "rf__max_depth": [10, None]
}

# GridSearchCV
smote_search = GridSearchCV(
    smote_pipeline,
    smote_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

# Train model
smote_search.fit(X_train, y_train)

# Predict on original test data
smote_pred = smote_search.predict(X_test)

print("\nBest Parameters:")
print(smote_search.best_params_)

print("\nBest CV F1 Score:")
print(smote_search.best_score_)

print("\nTest Accuracy:")
print(accuracy_score(y_test, smote_pred))

print("\nClassification Report:")
print(classification_report(y_test, smote_pred))

Original Training Class Distribution:
Sentiment
1    6904
0    1096
Name: count, dtype: int64

Best Parameters:
{'rf__max_depth': 10, 'rf__n_estimators': 100}

Best CV F1 Score:
0.9688078658542035

Test Accuracy:
0.9535

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.79      0.82       274
           1       0.97      0.98      0.97      1726

    accuracy                           0.95      2000
   macro avg       0.91      0.89      0.90      2000
weighted avg       0.95      0.95      0.95      2000



# Task 4 – Compare Class Weights Before and After

In [7]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

# Model without class weights
svm_before = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=2000)),
    ("svm", SVC(kernel="linear"))
])

svm_before.fit(X_train, y_train)

pred_before = svm_before.predict(X_test)

# Model with balanced class weights
svm_after = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=2000)),
    ("svm", SVC(kernel="linear", class_weight="balanced"))
])

svm_after.fit(X_train, y_train)

pred_after = svm_after.predict(X_test)

# Compare results
results = pd.DataFrame({
    "Model": ["Without Class Weights", "With Balanced Class Weights"],
    "Accuracy": [
        accuracy_score(y_test, pred_before),
        accuracy_score(y_test, pred_after)
    ],
    "F1 Score": [
        f1_score(y_test, pred_before),
        f1_score(y_test, pred_after)
    ]
})

print(results)

                         Model  Accuracy  F1 Score
0        Without Class Weights    0.9630  0.978748
1  With Balanced Class Weights    0.9155  0.949416


# Task 5 – AI-Suggested GridSearchCV Parameter Grid

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

# Create Random Forest pipeline
rf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=1500)),
    ("rf", RandomForestClassifier(random_state=42))
])

# AI-suggested parameter grid
param_grid = {
    "rf__n_estimators": [50, 100],
    "rf__max_depth": [10, 20, None],
    "rf__min_samples_split": [2, 5],
    "rf__min_samples_leaf": [1, 2]
}

# GridSearchCV
ai_grid = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

# Train model
ai_grid.fit(X_train, y_train)

# Best model prediction
best_rf_model = ai_grid.best_estimator_
ai_pred = best_rf_model.predict(X_test)

print("Best Parameters:")
print(ai_grid.best_params_)

print("\nBest CV F1 Score:")
print(ai_grid.best_score_)

print("\nTest Accuracy:")
print(accuracy_score(y_test, ai_pred))

print("\nClassification Report:")
print(classification_report(y_test, ai_pred))

Best Parameters:
{'rf__max_depth': None, 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 2, 'rf__n_estimators': 50}

Best CV F1 Score:
0.9780713836189318

Test Accuracy:
0.9625

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.81      0.86       274
           1       0.97      0.99      0.98      1726

    accuracy                           0.96      2000
   macro avg       0.94      0.90      0.92      2000
weighted avg       0.96      0.96      0.96      2000

